# Retrain Triple-Barrier Transformer

Load a trained triple-barrier Transformer checkpoint, fine-tune it on 15-minute labelled feature data for one or two selected stocks, save a versioned checkpoint, and run one inference example.

In [1]:
from datetime import datetime
from pathlib import Path
import random
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PACKAGE_SRC = PROJECT_ROOT / "packages" / "src"
if str(PACKAGE_SRC) not in sys.path:
    sys.path.insert(0, str(PACKAGE_SRC))

PROJECT_ROOT

PosixPath('/Users/akash/PycharmProjects/AlgoTrade')

In [2]:
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split

from tradingbot.models import TransformerWindowDataset, build_transformer

DATA_DIR = PROJECT_ROOT / "data" / "historic_candle_feature"
MODEL_DIR = PROJECT_ROOT / "data" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

BASE_CHECKPOINT_PATH = MODEL_DIR / "triple_barrier_transformer.pt"  # set None to auto-pick the latest versioned base model
BASE_CHECKPOINT_GLOB = "triple_barrier_transformer_classification_*_valacc_*.pt"
CHECKPOINT_PREFIX = "triple_barrier_transformer"

TIMEFRAME = "15minute"
RETRAIN_STOCKS = ["AXISBANK"]  # choose one or two symbols, e.g. ["AXISBANK", "HDFCBANK"]

SEED = 42
VALIDATION_FRACTION = 0.2
BATCH_SIZE = 64
EPOCHS = 20
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-2
MAX_ROWS_PER_FILE = None

LABEL_COLUMNS = [
    "triple_barrier_label_lower",
    "triple_barrier_label_neutral",
    "triple_barrier_label_upper",
]
REGRESSION_COLUMNS = ["triple_barrier_return"]

random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

DEVICE

device(type='mps')

In [3]:
def resolve_base_checkpoint():
    if BASE_CHECKPOINT_PATH is not None and Path(BASE_CHECKPOINT_PATH).exists():
        return Path(BASE_CHECKPOINT_PATH)
    candidates = sorted(
        MODEL_DIR.glob(BASE_CHECKPOINT_GLOB),
        key=lambda path: path.stat().st_mtime,
    )
    if not candidates:
        raise FileNotFoundError(
            f"No base checkpoint found. Checked {BASE_CHECKPOINT_PATH} and {MODEL_DIR / BASE_CHECKPOINT_GLOB}"
        )
    return candidates[-1]


base_checkpoint_path = resolve_base_checkpoint()
base_checkpoint = torch.load(base_checkpoint_path, map_location=DEVICE)
base_model_config = base_checkpoint["model_config"]
feature_columns = base_model_config.get("feature_columns") or base_checkpoint.get("feature_columns")
if not feature_columns:
    raise ValueError("Base checkpoint does not include feature_columns; retraining cannot align the dataset.")

OUTPUT_TYPE = base_model_config["output_type"]
CONTEXT_LENGTH = base_checkpoint.get("context_length", 64)
TARGET_OFFSET = base_checkpoint.get("target_offset", 0)

if OUTPUT_TYPE == "classification":
    target_mode = "class_index"
    target_kwargs = {"label_columns": base_checkpoint.get("label_columns") or LABEL_COLUMNS}
    output_dim = len(target_kwargs["label_columns"])
else:
    target_mode = "regression"
    target_kwargs = {"target_columns": base_checkpoint.get("target_columns") or REGRESSION_COLUMNS}
    output_dim = len(target_kwargs["target_columns"])

{
    "base_checkpoint": base_checkpoint_path.name,
    "objective_type": OUTPUT_TYPE,
    "context_length": CONTEXT_LENGTH,
    "target_offset": TARGET_OFFSET,
    "feature_count": len(feature_columns),
}

{'base_checkpoint': 'triple_barrier_transformer.pt',
 'objective_type': 'classification',
 'context_length': 128,
 'target_offset': 0,
 'feature_count': 21}

In [4]:
if not 1 <= len(RETRAIN_STOCKS) <= 2:
    raise ValueError("Choose one or two stocks in RETRAIN_STOCKS.")

matched_files = []
missing_stocks = []
for stock in RETRAIN_STOCKS:
    matches = sorted(DATA_DIR.glob(f"{stock}_{TIMEFRAME}_*_features_triple_barrier.csv"))
    if matches:
        matched_files.extend(matches)
    else:
        missing_stocks.append(stock)

if missing_stocks:
    raise FileNotFoundError(f"No {TIMEFRAME} labelled feature files found for: {missing_stocks}")

schema = pd.read_csv(matched_files[0], nrows=0).columns.tolist()
missing_features = [column for column in feature_columns if column not in schema]
required_targets = target_kwargs.get("label_columns") or target_kwargs.get("target_columns") or []
missing_targets = [column for column in required_targets if column not in schema]
if missing_features or missing_targets:
    raise ValueError(
        f"Retrain files do not match the base checkpoint schema. Missing features={missing_features}, missing targets={missing_targets}"
    )

pd.DataFrame({"matched_file": [path.name for path in matched_files]})

,matched_file
0,AXISBANK_15minute_1990-01-01T00-00-00plus05-30...


In [5]:
full_dataset = TransformerWindowDataset(
    matched_files,
    feature_columns=feature_columns,
    context_length=CONTEXT_LENGTH,
    target_mode=target_mode,
    target_offset=TARGET_OFFSET,
    max_rows_per_file=MAX_ROWS_PER_FILE,
    **target_kwargs,
)
if len(full_dataset) == 0:
    raise ValueError(f"Retraining dataset is empty. Skipped files: {full_dataset.skipped[:5]}")

val_size = max(1, int(len(full_dataset) * VALIDATION_FRACTION))
train_size = len(full_dataset) - val_size
if train_size <= 0:
    raise ValueError("Not enough retraining windows for a train/validation split.")

generator = torch.Generator().manual_seed(SEED)
train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size],
    generator=generator,
)

pin_memory = DEVICE.type == "cuda"
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=pin_memory,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=pin_memory,
)

{
    "total_windows": len(full_dataset),
    "train_windows": len(train_dataset),
    "val_windows": len(val_dataset),
    "label_counts": None if full_dataset.label_counts is None else full_dataset.label_counts.tolist(),
    "skipped": full_dataset.skipped[:5],
}

{'total_windows': 69312,
 'train_windows': 55450,
 'val_windows': 13862,
 'label_counts': [36521, 280, 32511],
 'skipped': []}

In [6]:
builder_config = {
    key: value
    for key, value in base_model_config.items()
    if key not in {"feature_columns", "file_glob", "timeframe_file_pattern", "matched_files"}
}
model = build_transformer(**builder_config).to(DEVICE)
model.load_state_dict(base_checkpoint["model_state_dict"])

if OUTPUT_TYPE == "classification":
    class_weight = None
    if full_dataset.label_counts is not None and (full_dataset.label_counts > 0).all():
        counts = torch.tensor(full_dataset.label_counts, dtype=torch.float32, device=DEVICE)
        class_weight = counts.sum() / (counts.numel() * counts)
    criterion = nn.CrossEntropyLoss(weight=class_weight)
else:
    criterion = nn.MSELoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

sum(parameter.numel() for parameter in model.parameters())

30819

In [7]:
def move_batch(batch, device):
    x, y = batch
    return x.to(device), y.to(device)


def classification_metrics(correct, total, true_positive, false_positive, false_negative):
    precision = true_positive / (true_positive + false_positive).clamp_min(1)
    recall = true_positive / (true_positive + false_negative).clamp_min(1)
    f1 = 2 * precision * recall / (precision + recall).clamp_min(1e-12)
    return {
        "accuracy": correct / max(total, 1),
        "precision_macro": precision.mean().item(),
        "recall_macro": recall.mean().item(),
        "f1_macro": f1.mean().item(),
        "precision_by_class": precision.cpu().tolist(),
        "recall_by_class": recall.cpu().tolist(),
        "f1_by_class": f1.cpu().tolist(),
    }


def checkpoint_path_for(objective_type, final_val_accuracy):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    accuracy_text = "na" if final_val_accuracy is None else f"{final_val_accuracy:.4f}"
    stock_text = "-".join(RETRAIN_STOCKS)
    return MODEL_DIR / f"{CHECKPOINT_PREFIX}_{objective_type}_{stock_text}_{TIMEFRAME}_{timestamp}_valacc_{accuracy_text}.pt"


def train_one_epoch(model, loader):
    model.train()
    total_loss = 0.0
    total_items = 0

    for batch in loader:
        x, y = move_batch(batch, DEVICE)
        optimizer.zero_grad(set_to_none=True)
        prediction = model(x)
        loss = criterion(prediction, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        batch_size = x.size(0)
        total_loss += loss.item() * batch_size
        total_items += batch_size

    return total_loss / max(total_items, 1)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    total_items = 0

    if OUTPUT_TYPE == "classification":
        class_count = output_dim
        correct = 0
        true_positive = torch.zeros(class_count, dtype=torch.float32, device=DEVICE)
        false_positive = torch.zeros(class_count, dtype=torch.float32, device=DEVICE)
        false_negative = torch.zeros(class_count, dtype=torch.float32, device=DEVICE)

    for batch in loader:
        x, y = move_batch(batch, DEVICE)
        prediction = model(x)
        loss = criterion(prediction, y)

        batch_size = x.size(0)
        total_loss += loss.item() * batch_size
        total_items += batch_size

        if OUTPUT_TYPE == "classification":
            predicted = prediction.argmax(dim=-1)
            correct += (predicted == y).sum().item()
            for class_index in range(class_count):
                predicted_class = predicted == class_index
                actual_class = y == class_index
                true_positive[class_index] += (predicted_class & actual_class).sum()
                false_positive[class_index] += (predicted_class & ~actual_class).sum()
                false_negative[class_index] += (~predicted_class & actual_class).sum()

    metrics = {"loss": total_loss / max(total_items, 1)}
    if OUTPUT_TYPE == "classification":
        metrics.update(
            classification_metrics(correct, total_items, true_positive, false_positive, false_negative)
        )
    return metrics


def prefixed_metrics(prefix, metrics):
    return {f"{prefix}_{key}": value for key, value in metrics.items()}


history = []

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader)
    train_metrics = evaluate(model, train_loader)
    val_metrics = evaluate(model, val_loader)
    row = {
        "epoch": epoch,
        "train_optimized_loss": train_loss,
        **prefixed_metrics("train", train_metrics),
        **prefixed_metrics("val", val_metrics),
    }
    history.append(row)
    print(row)

final_metrics = history[-1]
final_val_accuracy = final_metrics.get("val_accuracy")
saved_checkpoint_path = checkpoint_path_for(OUTPUT_TYPE, final_val_accuracy)
model_config = {
    **base_model_config,
    "feature_columns": feature_columns,
    "retrain_stocks": RETRAIN_STOCKS,
    "timeframe": TIMEFRAME,
    "matched_files": [path.name for path in matched_files],
    "base_checkpoint": base_checkpoint_path.name,
}
checkpoint_context = {
    "model_state_dict": model.state_dict(),
    "model_config": model_config,
    "feature_columns": feature_columns,
    "label_columns": target_kwargs.get("label_columns"),
    "target_columns": target_kwargs.get("target_columns"),
    "context_length": CONTEXT_LENGTH,
    "target_offset": TARGET_OFFSET,
    "history": history,
    "final_metrics": final_metrics,
    "objective_type": OUTPUT_TYPE,
    "base_checkpoint": base_checkpoint_path.name,
    "retrain_stocks": RETRAIN_STOCKS,
    "timeframe": TIMEFRAME,
    "matched_files": [path.name for path in matched_files],
    "saved_checkpoint_name": saved_checkpoint_path.name,
}
torch.save(checkpoint_context, saved_checkpoint_path)
print(f"Saved retrained model to {saved_checkpoint_path}")

pd.DataFrame(history)

{'epoch': 1, 'train_optimized_loss': 1.434531275366103, 'train_loss': 1.3491666340698951, 'train_accuracy': 0.5232281334535618, 'train_precision_macro': 0.34234490990638733, 'train_recall_macro': 0.34215298295021057, 'train_f1_macro': 0.3312370181083679, 'train_precision_by_class': [0.5373283624649048, 0.0, 0.489706426858902], 'train_recall_by_class': [0.7165846824645996, 0.0, 0.30987435579299927], 'train_f1_by_class': [0.6141434907913208, 0.0, 0.37956756353378296], 'val_loss': 1.1886535963271194, 'val_accuracy': 0.5254653008223922, 'val_precision_macro': 0.34578457474708557, 'val_recall_macro': 0.34466519951820374, 'val_f1_macro': 0.33338236808776855, 'val_precision_by_class': [0.5352790355682373, 0.0, 0.5020747184753418], 'val_recall_by_class': [0.72066730260849, 0.0, 0.31332826614379883], 'val_f1_by_class': [0.614290714263916, 0.0, 0.3858563303947449]}
{'epoch': 2, 'train_optimized_loss': 1.2968342342153125, 'train_loss': 1.2785942198952862, 'train_accuracy': 0.5405049594229036, 'tr

,epoch,train_optimized_loss,train_loss,train_accuracy,train_precision_macro,train_recall_macro,train_f1_macro,train_precision_by_class,train_recall_by_class,train_f1_by_class,val_loss,val_accuracy,val_precision_macro,val_recall_macro,val_f1_macro,val_precision_by_class,val_recall_by_class,val_f1_by_class
0,1,1.434531,1.349167,0.523228,0.342345,0.342153,0.331237,"[0.5373283624649048, 0.0, 0.489706426858902]","[0.7165846824645996, 0.0, 0.30987435579299927]","[0.6141434907913208, 0.0, 0.37956756353378296]",1.188654,0.525465,0.345785,0.344665,0.333382,"[0.5352790355682373, 0.0, 0.5020747184753418]","[0.72066730260849, 0.0, 0.31332826614379883]","[0.614290714263916, 0.0, 0.3858563303947449]"
1,2,1.296834,1.278594,0.540505,0.356414,0.351957,0.335417,"[0.5464441180229187, 0.0, 0.5227974653244019]","[0.7752494215965271, 0.0, 0.2806212902069092]","[0.6410419344902039, 0.0, 0.36520954966545105]",1.161228,0.537513,0.356130,0.351059,0.332822,"[0.5406954288482666, 0.0, 0.5276959538459778]","[0.7803667187690735, 0.0, 0.2728103697299957]","[0.6387901306152344, 0.0, 0.3596746623516083]"
2,3,1.289787,1.240632,0.551434,0.364930,0.361176,0.351447,"[0.5573757886886597, 0.0, 0.5374128818511963]","[0.7416632771492004, 0.0, 0.34186387062072754]","[0.636447548866272, 0.0, 0.41789355874061584]",1.123874,0.539388,0.357046,0.353925,0.342846,"[0.5447959303855896, 0.0, 0.5263416767120361]","[0.73610919713974, 0.0, 0.3256664276123047]","[0.626165509223938, 0.0, 0.4023713171482086]"
3,4,1.291945,1.201008,0.547574,0.365114,0.366779,0.365584,"[0.5778396725654602, 0.0, 0.5175031423568726]","[0.5459546446800232, 0.0, 0.5543822050094604]","[0.5614448189735413, 0.0, 0.5353082418441772]",1.106656,0.539893,0.359870,0.361096,0.360290,"[0.5641173124313354, 0.0, 0.5154937505722046]","[0.5410175323486328, 0.0, 0.5422695875167847]","[0.5523260235786438, 0.0, 0.5285427570343018]"
4,5,1.265019,1.228958,0.585338,0.391654,0.384139,0.376028,"[0.5820648670196533, 0.0, 0.5928976535797119]","[0.7695435285568237, 0.0, 0.3828721046447754]","[0.6628018021583557, 0.0, 0.4652818441390991]",1.143643,0.572068,0.383223,0.375929,0.366323,"[0.5680630803108215, 0.0, 0.5816052556037903]","[0.7646490931510925, 0.0, 0.3631378412246704]","[0.6518570184707642, 0.0, 0.44711175560951233]"
5,6,1.256636,1.201287,0.597475,0.399204,0.393492,0.388287,"[0.5948499441146851, 0.0, 0.6027605533599854]","[0.7529725432395935, 0.0, 0.4275032877922058]","[0.6646360754966736, 0.0, 0.5002254843711853]",1.148447,0.577262,0.385394,0.380534,0.374372,"[0.5756564736366272, 0.0, 0.5805251598358154]","[0.7374879121780396, 0.0, 0.40411272644996643]","[0.6466001272201538, 0.0, 0.476515531539917]"
6,7,1.230091,1.180652,0.611794,0.429444,0.398119,0.379011,"[0.5887575745582581, 0.0, 0.699574887752533]","[0.8835588097572327, 0.0, 0.31079936027526855]","[0.7066441774368286, 0.0, 0.4303898811340332]",1.129939,0.591257,0.413675,0.385587,0.362778,"[0.5718277096748352, 0.0, 0.6691973805427551]","[0.8748103976249695, 0.0, 0.2819497287273407]","[0.6915907859802246, 0.0, 0.396742045879364]"
7,8,1.210175,1.209694,0.620487,0.525232,0.411777,0.410494,"[0.6186213493347168, 0.3333333432674408, 0.623...","[0.7414240837097168, 0.0042372881434857845, 0....","[0.6744785904884338, 0.008368200622498989, 0.5...",1.158103,0.601284,0.401542,0.397788,0.394536,"[0.5988425016403198, 0.0, 0.6057844758033752]","[0.7275609970092773, 0.0, 0.46580350399017334]","[0.656956136226654, 0.0, 0.52665114402771]"
8,9,1.189763,1.123807,0.634968,0.599484,0.421173,0.417640,"[0.6154295206069946, 0.5, 0.6830226182937622]","[0.8285841345787048, 0.012711863964796066, 0.4...","[0.706274688243866, 0.024793386459350586, 0.52...",1.129044,0.606550,0.412161,0.398682,0.389137,"[0.5911841988563538, 0.0, 0.6453002095222473]","[0.8080794215202332, 0.0, 0.38796648383140564]","[0.6828216910362244, 0.0, 0.4845890402793884]"
9,10,1.156923,1.093295,0.640000,0.570298,0.428143,0.430204,"[0.6364902853965759, 0.4285714328289032, 0.645...","[0.7492483258247375, 0.012711863964796066, 0.5

In [8]:
checkpoint = torch.load(saved_checkpoint_path, map_location=DEVICE)
checkpoint_model_config = checkpoint["model_config"]
checkpoint_builder_config = {
    key: value
    for key, value in checkpoint_model_config.items()
    if key not in {"feature_columns", "file_glob", "timeframe_file_pattern", "matched_files", "retrain_stocks", "timeframe", "base_checkpoint"}
}
inference_model = build_transformer(**checkpoint_builder_config).to(DEVICE)
inference_model.load_state_dict(checkpoint["model_state_dict"])
inference_model.eval()

x, y = val_dataset[0]
with torch.no_grad():
    output = inference_model(x.unsqueeze(0).to(DEVICE))

if checkpoint_model_config["output_type"] == "classification":
    probabilities = torch.softmax(output, dim=-1).squeeze(0).cpu()
    predicted_index = int(probabilities.argmax().item())
    labels = checkpoint["label_columns"]
    inference_result = {
        "checkpoint": saved_checkpoint_path.name,
        "base_checkpoint": checkpoint["base_checkpoint"],
        "retrain_stocks": checkpoint["retrain_stocks"],
        "timeframe": checkpoint["timeframe"],
        "predicted_label": labels[predicted_index],
        "probabilities": dict(zip(labels, probabilities.tolist())),
        "actual_index": int(y.item()),
        "actual_label": labels[int(y.item())],
    }
else:
    prediction = output.squeeze(0).cpu().tolist()
    actual = y.cpu().tolist()
    inference_result = {
        "checkpoint": saved_checkpoint_path.name,
        "base_checkpoint": checkpoint["base_checkpoint"],
        "retrain_stocks": checkpoint["retrain_stocks"],
        "timeframe": checkpoint["timeframe"],
        "prediction": dict(zip(checkpoint["target_columns"], prediction)),
        "actual": dict(zip(checkpoint["target_columns"], actual)),
    }

inference_result

{'checkpoint': 'triple_barrier_transformer_classification_AXISBANK_15minute_20260516_022110_valacc_0.6600.pt',
 'base_checkpoint': 'triple_barrier_transformer.pt',
 'retrain_stocks': ['AXISBANK'],
 'timeframe': '15minute',
 'predicted_label': 'triple_barrier_label_lower',
 'probabilities': {'triple_barrier_label_lower': 0.6293863654136658,
  'triple_barrier_label_neutral': 0.003395744366571307,
  'triple_barrier_label_upper': 0.3672179579734802},
 'actual_index': 2,
 'actual_label': 'triple_barrier_label_upper'}